# 09 - Maintain Fabric Delta tables

Deletes old uncommitted attempt output, compacts recent partitions, and vacuums files according to an approved retention policy. Run only after confirming that replay, audit, privacy, and legal-hold requirements permit the configured retention.

**Before running:** In the Lakehouse explorer, attach and pin `people_counter_<environment>` as this notebook's default Lakehouse.

In [ ]:
DATABASE = ""
TABLE_PREFIX = "people_counter"
UNCOMMITTED_RETENTION_DAYS = 30
OPTIMIZE_LOOKBACK_DAYS = 7
VACUUM_RETENTION_HOURS = 168
RUN_VACUUM = False

In [ ]:
from datetime import date, datetime, timedelta, timezone
import json
import re

import notebookutils
from delta.tables import DeltaTable
from pyspark.sql import SparkSession, functions as F


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def parameter_bool(value: object, name: str) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, str) and value.strip().lower() in {"true", "false"}:
        return value.strip().lower() == "true"
    raise ValueError(f"{name} must be true or false")


run_vacuum = parameter_bool(RUN_VACUUM, "RUN_VACUUM")
database = DATABASE.strip()
prefix = TABLE_PREFIX.strip()
if database and IDENTIFIER.fullmatch(database) is None:
    raise ValueError("DATABASE is not a valid identifier")
if IDENTIFIER.fullmatch(prefix) is None:
    raise ValueError("TABLE_PREFIX is not a valid identifier")
uncommitted_days = int(UNCOMMITTED_RETENTION_DAYS)
optimize_days = int(OPTIMIZE_LOOKBACK_DAYS)
vacuum_hours = int(VACUUM_RETENTION_HOURS)
if uncommitted_days < 1 or optimize_days < 1 or vacuum_hours < 168:
    raise ValueError("Retention must be positive and VACUUM_RETENTION_HOURS must be at least 168")


def table(suffix: str) -> str:
    value = f"{prefix}_{suffix}"
    return f"{database}.{value}" if database else value


def sql_table(suffix: str) -> str:
    value = f"{prefix}_{suffix}"
    return f"`{database}`.`{value}`" if database else f"`{value}`"


spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")
now = datetime.now(timezone.utc)
uncommitted_cutoff = now - timedelta(days=uncommitted_days)
optimize_start = (now - timedelta(days=optimize_days)).date()
work = spark_session.table(table("video_work")).select(
    "work_id",
    "committed_attempt_id",
)
stale_attempts = (
    spark_session.table(table("video_attempts"))
    .where(
        F.col("completed_at").isNotNull()
        & (F.col("completed_at") < F.lit(uncommitted_cutoff))
    )
    .join(work, "work_id", "left")
    .where(F.col("committed_attempt_id").isNull() | (F.col("attempt_id") != F.col("committed_attempt_id")))
    .select("work_id", "attempt_id", "capture_date")
    .dropDuplicates()
    .cache()
)
stale_attempt_count = stale_attempts.count()
for suffix in ("telemetry_attempts", "line_count_attempts"):
    (
        DeltaTable.forName(spark_session, table(suffix))
        .alias("t")
        .merge(
            stale_attempts.alias("s"),
            (
                "t.capture_date = s.capture_date AND t.work_id = s.work_id "
                "AND t.attempt_id = s.attempt_id"
            ),
        )
        .whenMatchedDelete()
        .execute()
    )

for suffix in (
    "video_work",
    "video_attempts",
    "telemetry_attempts",
    "line_count_attempts",
):
    spark_session.sql(
        f"OPTIMIZE {sql_table(suffix)} WHERE capture_date >= DATE '{optimize_start.isoformat()}'"
    )

if run_vacuum:
    for suffix in (
        "event_receipts",
        "video_work",
        "video_attempts",
        "dispatcher_leases",
        "replay_requests",
        "telemetry_attempts",
        "line_count_attempts",
        "reconciliation_findings",
        "processing_benchmarks",
        "gold_flow_minute",
        "gold_flow_hour",
        "gold_video",
        "gold_operations_hour",
    ):
        spark_session.sql(f"VACUUM {sql_table(suffix)} RETAIN {vacuum_hours} HOURS")

stale_attempts.unpersist()
outcome = {
    "maintained_at": now.isoformat(),
    "stale_uncommitted_attempts": stale_attempt_count,
    "optimize_start": optimize_start.isoformat(),
    "vacuum_ran": run_vacuum,
    "vacuum_retention_hours": vacuum_hours,
}
print(json.dumps(outcome, sort_keys=True))

In [ ]:
notebookutils.notebook.exit(json.dumps(outcome, sort_keys=True))